# Step 52 — SEISAN REA export with explicit archive composition

All Pinatubo-specific metadata and export policies are defined here. The generic writer remains the shared FLOVOpy `export_catalog_to_rea` library function. This notebook does not redefine Nordic serialization or require a separate step-52 module.

## Preserving W/P/H semantics

Read the trusted step-50 pickle. Each copied export event receives Nordic comment records for EVENT_CLASS, WAVEFORM_PRESENT, PICKS_PRESENT, HYPOCENTER_PRESENT and ORIGIN_STATUS. The original ObsPy catalogue remains unchanged. The seven composition classes include P_H (picks plus hypocenter without waveforms). Composition is not encoded as an earthquake-type code.

A real origin requires a timed origin with coordinates and no derived-origin marker. For an unlocated record, the export may need a time-only anchor for an S-file header: waveform-start metadata is preferred, otherwise earliest pick time. ORIGIN_STATUS=DERIVED and HYPOCENTER_PRESENT=False distinguish such anchors from located hypocenters. No artificial latitude/longitude/depth is assigned by this local policy.

## Existing files, collisions and inspection

Standard Nordic filenames have limited time resolution, so multiple distinct events can collide. `collision='error'` reports conflicts. Explicit `collision='shift'` allocates later **filename** seconds up to `max_shift_seconds`; actual origin and pick timestamps are unchanged. Filename-day counts can differ across midnight. The export manifest records allocations and verifies tracked-file reuse. Existing untracked files are not overwritten; `overwrite=True` permits updating this writer's tracked files only.

The run creates/reuses REA files under the configured external output root and may optionally write a SELECT file. Inspect written/reused/skipped/failed counts and manifest paths; a completed loop is not proof every event was exported. For older REA exports without the current manifest, use a new output location or resolve collisions explicitly. Run 53 afterward to audit the written database, and compare with the canonical catalogue manifest.

## Configuration

Inspect all input/output paths and settings. This cell resolves configuration; definitions and run cells follow.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    catalog_pkl=P['CATALOG_PREFIX'] + '.pkl',
    rea_dir=P['FAIR_REA_DIR'],
    author=P['DEFAULT_AUTHOR'],
    evtype=P['DEFAULT_EVTYPE'],
    nordic_format='OLD',
    write_select=False,
    collision="error",  # Or "shift" for standard-name collision allocation.
    max_shift_seconds=30,
    overwrite=False,
)
display(SETTINGS)

## Workflow implementation

The next cell defines functions only. It does not execute the workflow or write outputs.

In [ ]:
from __future__ import annotations

import os

import pickle

import hashlib

from pathlib import Path

from typing import List, Optional

from obspy import UTCDateTime

from obspy.core.event import Event, Comment, ResourceIdentifier

from flovopy.seisanio.rea_export import export_catalog_to_rea

from pinatubo_steps.shared import get_comment_value

def extract_wavefiles(event: Event) -> List[str]:
    wavs = []
    for c in (event.comments or []):
        txt = (c.text or "").strip()
        if txt.startswith("wavfile:"):
            wavs.append(os.path.basename(txt.split("wavfile:", 1)[1].strip()))
    return list(dict.fromkeys(wavs))

def extract_waveform_starttime(event: Event) -> Optional[UTCDateTime]:
    val = get_comment_value(event, "waveform_starttime:")
    if val:
        try:
            return UTCDateTime(val)
        except Exception:
            return None
    return None

def infer_event_class(event: Event) -> str:
    val = get_comment_value(event, "event_class:")
    return val if val else "UNKNOWN"

def infer_event_type(event: Event, default="L") -> str:
    val = get_comment_value(event, "mainclass:")
    return val if val else default

def _anchor_time(event):
    # Pinatubo policy, deliberately not a generic representative-event-time API.
    time = extract_waveform_starttime(event)
    if time is not None:
        return time, 'waveform_starttime'
    times = [p.time for p in event.picks if p.time is not None]
    return (min(times), 'earliest_pick') if times else None

def _prepare_event(event):
    timed = [o for o in event.origins if o.time is not None]
    real = bool(timed) and not get_comment_value(event, 'origin_derived_from:') and any(
        o.latitude is not None and o.longitude is not None for o in timed)
    if not real:
        for origin in event.origins:
            origin.latitude = origin.longitude = origin.depth = None
    prefixes = ('EVENT_CLASS:', 'WAVEFORM_PRESENT:', 'PICKS_PRESENT:',
                'HYPOCENTER_PRESENT:', 'ORIGIN_STATUS:')
    event.comments = [c for c in event.comments if not (c.text or '').startswith(prefixes)]
    values = dict(EVENT_CLASS=infer_event_class(event),
        WAVEFORM_PRESENT=bool(extract_wavefiles(event)), PICKS_PRESENT=bool(event.picks),
        HYPOCENTER_PRESENT=real, ORIGIN_STATUS='REAL' if real else 'DERIVED')
    event.comments.extend(Comment(text=f'{key}:{value}', resource_id=ResourceIdentifier(
        'smi:local/pinatubo/export-comment/' + hashlib.sha256(
            (str(event.resource_id)+'|'+key).encode()).hexdigest())) for key, value in values.items())
    return event

def run(*, catalog_pkl, rea_dir, author='GT__', evtype='L', nordic_format='OLD',
        write_select=False, collision='error', max_shift_seconds=30, overwrite=False):
    """Export Step-50 catalog, reporting writes/reuse/failures instead of suffixing names.

    collision='shift' explicitly permits filename-only shifts for same-second events;
    actual origin and pick times are never shifted. Default collisions are failures.
    """
    with open(catalog_pkl, 'rb') as stream:
        catalog = pickle.load(stream)
    result = export_catalog_to_rea(
        catalog, rea_dir, author=author, event_type=lambda e: infer_event_type(e, evtype),
        nordic_format=nordic_format, high_accuracy=False,
        prepare_event=_prepare_event, waveform_files=extract_wavefiles,
        anchor_time=_anchor_time, collision=collision, max_shift_seconds=max_shift_seconds,
        overwrite=overwrite, write_select=write_select,
    )
    print('STEP 52:', {name: len(result[name]) for name in ('written','reused','skipped','failed')})
    if result['failed']:
        print('Export failures:', result['failed'])
    return result

## Run and inspect

Run this cell only after the configuration and all definitions. It writes the configured outputs; inspect the summary and exceptions before proceeding.

In [ ]:
result = run(**SETTINGS)

In [ ]:
display({key: len(result[key]) for key in ('written', 'reused', 'skipped', 'failed')})
display(result['rows'][:20])
display(result['failed'])
print('Manifest:', result['manifest_path'])